# 01 — Stream an NWB recording from DANDI

This notebook opens an extracellular electrophysiology recording directly from DANDI without downloading the complete NWB file. It resolves the public asset URL, discovers the available `ElectricalSeries`, creates a lazy SpikeInterface recording extractor, and loads only a short trace segment for inspection.

**Dandiset:** [000231, version 0.220904.1554](https://dandiarchive.org/dandiset/000231/0.220904.1554)  
**Asset:** `sub-KM131/sub-KM131_ses-20180116T184757_behavior+ecephys+image.nwb`  
**Asset size:** approximately 5.48 GB

> Streaming is lazy, but it still transfers the byte ranges required by each operation. Avoid requesting the complete recording with `get_traces()` and no time limits.

## 1. Imports

In [ ]:
from contextlib import closing

import h5py
import matplotlib.pyplot as plt
import numpy as np
import remfile
import spikeinterface.full as si
from dandi.dandiapi import DandiAPIClient

print(f"SpikeInterface version: {si.__version__}")

## 2. Identify the DANDI asset

The asset UUID comes from the DANDI/Neurosift URL. Resolving it through the DANDI API avoids hard-coding a temporary signed storage URL.

In [ ]:
DANDISET_ID = "000231"
DANDISET_VERSION = "0.220904.1554"
ASSET_ID = "8b5b9aa5-69d7-4ba8-a033-12b8450e2bca"
EXPECTED_ASSET_PATH = (
    "sub-KM131/"
    "sub-KM131_ses-20180116T184757_behavior+ecephys+image.nwb"
)

In [ ]:
with DandiAPIClient() as client:
    asset = client.get_asset(ASSET_ID)
    asset_path = asset.path
    asset_size_bytes = asset.size
    streaming_url = asset.get_content_url(
        follow_redirects=1,
        strip_query=True,
    )

assert asset_path == EXPECTED_ASSET_PATH

print(f"Dandiset: {DANDISET_ID}/{DANDISET_VERSION}")
print(f"Asset path: {asset_path}")
print(f"Asset size: {asset_size_bytes / 1e9:.2f} GB")
print(f"Streaming host: {streaming_url.split('/')[2]}")

## 3. Discover the `ElectricalSeries`

An NWB file can contain more than one electrical stream. The next cell inspects the remote HDF5 structure and finds groups whose NWB type is `ElectricalSeries`. It reads metadata and structure only; it does not load the complete voltage array.

In [ ]:
def decode_hdf5_attribute(value):
    """Convert byte-valued HDF5 attributes to ordinary strings."""
    return value.decode() if isinstance(value, bytes) else value


def find_electrical_series(h5_file):
    """Return the paths of all NWB ElectricalSeries groups."""
    paths = []

    def visitor(name, obj):
        neurodata_type = decode_hdf5_attribute(
            obj.attrs.get("neurodata_type")
        )
        if neurodata_type == "ElectricalSeries":
            paths.append(name)

    h5_file.visititems(visitor)
    return paths

In [ ]:
with closing(remfile.File(streaming_url)) as remote_file:
    with h5py.File(remote_file, mode="r") as h5_file:
        electrical_series_paths = find_electrical_series(h5_file)

if not electrical_series_paths:
    raise RuntimeError("No ElectricalSeries was found in the NWB asset.")

print("Available ElectricalSeries:")
for path in electrical_series_paths:
    print(f"- {path}")

## 4. Create a lazy SpikeInterface recording

If the file contains multiple `ElectricalSeries`, inspect the printed paths and select the raw high-frequency extracellular recording explicitly.

In [ ]:
ELECTRICAL_SERIES_PATH = electrical_series_paths[0]

recording = si.read_nwb_recording(
    file_path=streaming_url,
    electrical_series_path=ELECTRICAL_SERIES_PATH,
    stream_mode="remfile",
)

recording

## 5. Inspect recording metadata

In [ ]:
sampling_frequency = recording.get_sampling_frequency()
num_channels = recording.get_num_channels()
num_segments = recording.get_num_segments()
duration_s = recording.get_total_duration()

print(f"ElectricalSeries: {ELECTRICAL_SERIES_PATH}")
print(f"Sampling frequency: {sampling_frequency:,.1f} Hz")
print(f"Channels: {num_channels}")
print(f"Segments: {num_segments}")
print(f"Total duration: {duration_s / 60:.2f} min")
print(f"Data type: {recording.get_dtype()}")
print(f"Channel IDs: {recording.get_channel_ids()}")

## 6. Stream and plot a short trace segment

Start with a small interval and a subset of channels. Increase these values only after confirming that remote access works correctly.

In [ ]:
START_TIME_S = 10.0
DURATION_S = 2.0
MAX_CHANNELS_TO_PLOT = 8

selected_channel_ids = recording.get_channel_ids()[:MAX_CHANNELS_TO_PLOT]
recording_slice = recording.select_channels(selected_channel_ids)
recording_slice = recording_slice.time_slice(
    start_time=START_TIME_S,
    end_time=START_TIME_S + DURATION_S,
)

traces = recording_slice.get_traces(return_scaled=True)
times = np.arange(traces.shape[0]) / sampling_frequency + START_TIME_S

print(f"Loaded trace array: {traces.shape}")
print(f"Memory used: {traces.nbytes / 1e6:.2f} MB")

In [ ]:
fig, axes = plt.subplots(
    len(selected_channel_ids),
    1,
    figsize=(12, 1.5 * len(selected_channel_ids)),
    sharex=True,
)

axes = np.atleast_1d(axes)
for channel_index, (axis, channel_id) in enumerate(
    zip(axes, selected_channel_ids)
):
    axis.plot(times, traces[:, channel_index], linewidth=0.5)
    axis.set_ylabel(f"Ch {channel_id}\n(µV)")

axes[-1].set_xlabel("Time (s)")
fig.suptitle("Raw extracellular traces streamed from DANDI")
fig.tight_layout()
plt.show()

## Next step

The next notebook should inspect probe geometry and channel properties before applying preprocessing. Remote streaming is suitable for inspection and short slices, but full spike sorting repeatedly accesses the complete voltage signal. For that stage, downloading or locally caching the selected recording will usually be faster and more reliable.